In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [4]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [5]:
len(docs)

9

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [7]:
len(splitted_docs)

26

In [15]:
embeddings = OllamaEmbeddings(
    model = "nomic-embed-text"
)

In [17]:
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs, 
    embedding=embeddings
)

In [18]:
same_record = vector_store.similarity_search("Patient name")

#### agent = tools, llm, prompt

In [40]:
@tool
def retriever_tool(query : str) : 
    """
        This tool can help you to retrieve the relevant data of the PDF document and these 
        documents have details about medical reports. 
    """

    print("Tool called : ", query)

    docs = vector_store.similarity_search(query=query, k = 4)
    
    context = ""
    for doc in docs : 
        context += doc.page_content + "\n\n"


    return context

In [32]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

In [46]:
System_prompt = """
You are a helpful assistant that answers questions using the retrieved context.
ALWAYS use the `retriever_tool` for questions about the document.
For multi-part questions, call the tool separately for each part.
Answer ONLY from the retrieved context. If something is not in the context, say so.
Do not add outside medical advice.
"""

In [47]:
agent = create_agent(
    model = llm, 
    tools = [retriever_tool], 
    system_prompt=System_prompt
)

In [48]:
query = "What is the name of patient, and the what is the name of the doctors ? "

In [49]:
response = agent.invoke({"messages" : [{"role" : "user", "content" : query}]})

Tool called :  patient name doctor name


In [50]:
result = response["messages"][-1].content
print(result)

**Patient name (as shown in the report)**  
- **Ms. Nikita Chudhary**  

**Doctors named in the report**  
- **Dr. Nitin Nahar** – listed under “Reported” (referring physician)  
- **Dr. Swapnil Gupta**, MD, Pathology – Chief of Laboratory, Dr Lal PathLabs Ltd  
- **Dr. Kiran Bhargava Pathak**, MD, Pathology – Chief of Laboratory, Dr Lal PathLabs Ltd  
- **Dr. Sunanda**, MD, Pathology – Sr. Consultant Pathologist – Hematology & Immunology, Dr Lal PathLabs Ltd  
- **Dr. Beena Chandrasekhar**, PhD, Life Sciences – Technical Director – Flow Cytometry, Dr Lal PathLabs Ltd  
- **Dr. Nagarjun Sai Jaine**, MD, Pathology – Consultant – Hemato‑Oncopathology & Flow Cytometry, Dr Lal PathLabs Ltd  


In [51]:
query = "What are the test results and what did the doctor recommend?"
res = agent.invoke({"messages" : [{"role" : "user", "content" : query}]})
print(res["messages"][-1].content)

Tool called :  test results doctor recommended
**Test results (as shown in the excerpt)**  

- The report contains a “Test Report” section, but the actual numeric results for the individual tests are not included in the retrieved text.  
- The document does list general interpretive comments about vitamin‑D status:  

  *Decreased levels may be due to*  
  • Inadequate exposure to sunlight  
  • Dietary deficiency  
  • Vitamin‑D malabsorption  
  • Severe hepatocellular disease  
  • Certain drugs (e.g., anticonvulsants)  
  • Nephrotic syndrome  

  *Increased levels may indicate*  
  • Vitamin‑D intoxication  

**Doctor’s / laboratory recommendations (as stated in the document)**  

1. **Contact Customer Care if results are alarming or unexpected** – “If Test results are alarming or unexpected, client is advised to contact the Customer Care immediately for possible remedial action. Tel: +91‑11‑49885050.”  

2. **Clinical correlation is required** – “Laboratory investigations are onl